# Information Theory: Worked Solutions

In [ ]:
import numpy as np

def probability_vector(values):
    p = np.asarray(values, dtype=float)
    if p.ndim != 1 or p.size == 0 or not np.all(np.isfinite(p)) or np.any(p < 0):
        raise ValueError("finite nonnegative vector required")
    if not np.isclose(p.sum(), 1.0):
        raise ValueError("probabilities must sum to one")
    return p


## Problem 1. Entropy and zero-probability outcomes

**Problem.** Compute $H(p)=-\sum_k p_k\log p_k$ without evaluating $\log0$. Check a deterministic law, a fair bit, and $p=(1/2,1/4,1/8,1/8)$.

**Solution.** Restrict the sum to $p_k>0$; the omitted terms are zero by the limit $t\log t\to0$ as $t\downarrow0$. The answers are $0$, $\log2$, and $1.75\log2$ nats. Dividing by $\log2$ converts nats to bits. The helper below also checks that inputs are finite, nonnegative, and sum to one.

In [ ]:
def entropy(probabilities):
    p = probability_vector(probabilities)
    positive = p > 0
    return float(-np.sum(p[positive] * np.log(p[positive])))


In [ ]:
np.testing.assert_allclose(entropy([1.0, 0.0]), 0.0)
np.testing.assert_allclose(entropy([0.5, 0.5]), np.log(2))
p = np.array([0.5, 0.25, 0.125, 0.125])
np.testing.assert_allclose(entropy(p) / np.log(2), 1.75)
assert entropy(p) < np.log(p.size)


## Problem 2. Relative entropy and support mismatch

**Problem.** Evaluate $D_{\mathrm{KL}}(p\Vert q)$, including zero entries, and verify the cross-entropy identity and asymmetry with the distributions below.

**Solution.** For $p_k>0$, sum $p_k(\log p_k-\log q_k)$. If any such $q_k=0$, the corresponding cost is infinite. Zero-mass source entries contribute zero. For the preceding four-symbol $p$ and uniform $u$, $D_{\mathrm{KL}}(p\Vert u)=0.25\log2$ nats and

$$
-\sum_k p_k\log u_k=H(p)+D_{\mathrm{KL}}(p\Vert u).
$$

The reverse direction need not agree. For $(0.8,0.2)$ and $(0.5,0.5)$, the two divergences are approximately $0.1927$ and $0.2231$ nats. A positive probability floor would change the distributions and hence the problem; the implementation keeps infinite divergence when required.

In [ ]:
def kl_divergence(p, q):
    p, q = probability_vector(p), probability_vector(q)
    if p.shape != q.shape:
        raise ValueError("matching probability alphabets required")
    positive = p > 0
    if np.any(q[positive] == 0):
        return float("inf")
    return float(np.sum(p[positive] * (np.log(p[positive]) - np.log(q[positive]))))


In [ ]:
u = np.ones(4) / 4
np.testing.assert_allclose(kl_divergence(p, p), 0, atol=1e-14)
np.testing.assert_allclose(kl_divergence(p, u) / np.log(2), 0.25)
np.testing.assert_allclose(entropy(p) + kl_divergence(p, u), -p @ np.log(u))
assert np.isinf(kl_divergence([0.5, 0.5], [1.0, 0.0]))
np.testing.assert_allclose(kl_divergence([1.0, 0.0], [0.5, 0.5]), np.log(2))
forward = kl_divergence([0.8, 0.2], [0.5, 0.5])
reverse = kl_divergence([0.5, 0.5], [0.8, 0.2])
assert abs(forward - reverse) > 0.02
print({"forward_KL": forward, "reverse_KL": reverse})


## Problem 3. Mutual information from a joint table

**Problem.** Compute $I(X;Y)$ from a joint probability table. Check independent variables, a deterministic copy of a fair bit, and $Y=X\mathbin{\mathrm{xor}}N$, where $X$ is fair and $N\sim\operatorname{Bernoulli}(\varepsilon)$ is independent. What happens when the observed $Y$ is replaced by a constant?

**Solution.** Sum rows and columns to obtain $p_X$ and $p_Y$, form their outer product, and evaluate

$$
I(X;Y)=D_{\mathrm{KL}}(p_{X,Y}\Vert p_Xp_Y).
$$

Independence gives zero; an exact copy gives $\log2$ nats. For the noisy copy, the joint matrix is

$$
\frac12\begin{pmatrix}1-\varepsilon&\varepsilon\\\varepsilon&1-\varepsilon\end{pmatrix},
$$

so $I(X;Y)=\log2-h(\varepsilon)$, with $h(\varepsilon)$ the Bernoulli entropy. Both $\varepsilon=0$ and $\varepsilon=1$ reveal $X$ exactly; $\varepsilon=1/2$ gives independence. Replacing $Y$ by a constant makes the joint law factorize, giving zero mutual information.

In [ ]:
def mutual_information(joint):
    joint = np.asarray(joint, dtype=float)
    if joint.ndim != 2 or not np.all(np.isfinite(joint)) or np.any(joint < 0):
        raise ValueError("finite nonnegative joint matrix required")
    if not np.isclose(joint.sum(), 1):
        raise ValueError("joint probabilities must sum to one")
    independent = np.outer(joint.sum(axis=1), joint.sum(axis=0))
    return kl_divergence(joint.ravel(), independent.ravel())


In [ ]:
np.testing.assert_allclose(mutual_information(np.outer([0.3, 0.7], [0.2, 0.8])), 0, atol=1e-14)
np.testing.assert_allclose(mutual_information(np.diag([0.5, 0.5])), np.log(2))
for epsilon in (0.0, 0.1, 0.25, 0.5, 1.0):
    table = 0.5 * np.array([[1 - epsilon, epsilon], [epsilon, 1 - epsilon]])
    mi = mutual_information(table)
    np.testing.assert_allclose(mi, np.log(2) - entropy([epsilon, 1 - epsilon]), atol=1e-14)
    print({"flip_probability": epsilon, "mutual_information_bits": mi / np.log(2)})
# Coarsening an observation to a constant removes all mutual information.
np.testing.assert_allclose(mutual_information(np.array([[0.5], [0.5]])), 0, atol=1e-14)


## Problem 4. Stable log normalization

**Problem.** Evaluate $\operatorname{LSE}(a)=\log\sum_k e^{a_k}$ for finite log weights, including $a=(1000,999)$. Verify that adding a common shift of $\pm10^4$ leaves normalized log probabilities unchanged.

**Solution.** With $m=\max_k a_k$, factor out $e^m$ to obtain

$$
\operatorname{LSE}(a)=m+\log\sum_k e^{a_k-m}.
$$

All exponentials lie in $[0,1]$, and at least one is one. Thus $\operatorname{LSE}(1000,999)=1000+\log(1+e^{-1})\approx1000.3133$, while direct exponentiation overflows in ordinary floating-point arithmetic. Since $\operatorname{LSE}(a+c\mathbf1)=c+\operatorname{LSE}(a)$, the values $a_k-\operatorname{LSE}(a)$ are shift invariant. The checks allow small rounding errors. This function's input contract excludes infinite log weights.

In [ ]:
def logsumexp(values):
    a = np.asarray(values, dtype=float)
    if a.ndim != 1 or a.size == 0 or not np.all(np.isfinite(a)):
        raise ValueError("nonempty finite log-weight vector required")
    maximum = float(a.max())
    return maximum + float(np.log(np.exp(a - maximum).sum()))


In [ ]:
a = np.array([1000.0, 999.0])
np.testing.assert_allclose(logsumexp(a), 1000 + np.log1p(np.exp(-1)))
np.testing.assert_allclose(logsumexp([-1000.0, -1000.0]), -1000 + np.log(2))
base = np.array([-2.0, 0.0, 1.0])
logq = base - logsumexp(base)
for shift in (-10000.0, 10000.0):
    shifted = base + shift
    np.testing.assert_allclose(shifted - logsumexp(shifted), logq, atol=1e-11)
np.testing.assert_allclose(np.exp(logq).sum(), 1.0)
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.log(np.exp(a).sum())
assert np.isinf(naive) and np.isfinite(logsumexp(a))
print({"naive_result": naive, "stable_result": logsumexp(a), "normalized_probabilities": np.exp(logq)})


## Further problems: worked answers

**1. Why is cross-entropy at least entropy for a fixed source distribution? What happens under a support mismatch?**

For finite cross-entropy, $H(p,q)=H(p)+D_{\mathrm{KL}}(p\Vert q)$, and Gibbs' inequality gives $D_{\mathrm{KL}}\geq0$. Equality holds when $p=q$. If $p_k>0$ but $q_k=0$, both cross-entropy and KL divergence are infinite; the finite-alphabet source entropy remains finite.

**2. Why does a deterministic complemented bit contain as much information as an unchanged bit?**

Both maps are invertible: $Y=X$ gives $X=Y$, and $Y=1-X$ gives $X=1-Y$. In either case $H(X\mid Y)=0$, so $I(X;Y)=H(X)$. Whether bits match is not the same question as whether one determines the other.

**3. Would zero correlation alone make mutual information zero?**

No. Let $X$ be uniform on $\{-1,0,1\}$ and $Y=X^2$. Symmetry gives $E[X]=E[XY]=0$, hence zero covariance and correlation. Yet $Y$ is determined by $X$, and $I(X;Y)=H(Y)>0$ because $P(Y=1)=2/3$. Mutual information vanishes at independence, a stronger condition than zero correlation.

**4. Why can an exact mathematical shift remove overflow? Why does clipping a probability to a positive floor change the problem?**

The shift changes the intermediate values but preserves the exact algebraic expression: the common scale is accounted for by the added maximum. Floating-point overflow depends on intermediates, so this rearrangement matters. Clipping instead replaces entries of a probability vector and may require renormalization. It changes the law, potentially turning an infinite KL divergence into a finite one.

**5. A uniform density on an interval of length $1/4$ has differential entropy $-\log4$. Which discrete-entropy properties fail for this quantity?**

Differential entropy need not be nonnegative and is not invariant under an invertible rescaling. Densities may exceed one and depend on the coordinate unit. Here scaling $X$ by $8$ changes the interval length to $2$ and the entropy to $\log2$. Discrete outcome probabilities do not change under a one-to-one relabeling, so discrete entropy has different invariance properties.